# Individual Notebook: QingyiZhu

Name: QingyiZhu  
Selected area: Parramatta (`SA4_CODE = 125`, `Sydney - Parramatta`)  
Tasks included: Task 1 to Task 3.

This notebook keeps the common Task 1 cleaning workflow, includes Qingyi Zhu's Task 1.2 population-derived statistics, and then runs Task 2 and Task 3 for the Parramatta SA4 region.

# Task 1.1: Data cleaning

This common section loads the NSW region summary dataset and prepares a cleaned version for analysis. The main issue in this file is that many indicators are only available in selected years, so the cleaning keeps missing values during reshaping and only removes rows that have no usable year values.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

current_dir = Path.cwd()
data_path = current_dir / "data" / "Region summary_ New South Wales STE 1.csv"

df = pd.read_csv(data_path)
display(df.head())


## Cleaning logic

The raw table is a wide table, which means each row is one indicator and each year is stored in a separate column. To make the data easier to analyse, the cleaning step below does three things. First, it standardises the text column names and year column names. Second, it separates the measurement unit from the description column and records how many non-null years each indicator has. Third, it reshapes the table into a long format so that each row represents one indicator-year value.

In [ ]:
task1_df = df.rename(
    columns={
        "Measure Code": "measure_code",
        "Parent Description": "parent_description",
        "Description": "description",
    }
).copy()

task1_df.columns = [f"y_{col}" if str(col).isdigit() else col for col in task1_df.columns]
year_cols = [col for col in task1_df.columns if col.startswith("y_")]

task1_df["unit"] = task1_df["description"].str.extract(r"\(([^()]*)\)\s*$")
task1_df["description_clean"] = task1_df["description"].str.replace(r"\s*\([^()]*\)\s*$", "", regex=True)
task1_df["non_null_years"] = task1_df[year_cols].notna().sum(axis=1)
task1_df = task1_df[task1_df["non_null_years"] > 0].copy()

task1_long_df = task1_df.melt(
    id_vars=["measure_code", "parent_description", "description_clean", "unit"],
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)

task1_long_df = task1_long_df.dropna(subset=["value"]).copy()
task1_long_df["year"] = task1_long_df["year"].str.replace("y_", "").astype(int)
task1_analysis_df = (
    task1_long_df[task1_long_df["year"] != 2025]
    .drop_duplicates(subset=["measure_code", "description_clean", "year"])
    .copy()
)

cleaning_summary_df = pd.DataFrame(
    {
        "item": [
            "original_rows",
            "rows_after_drop_all_null_years",
            "long_table_rows",
            "analysis_rows_without_2025",
            "rows_with_only_one_non_null_year",
            "non_null_values_in_2025",
        ],
        "value": [
            len(df),
            len(task1_df),
            len(task1_long_df),
            len(task1_analysis_df),
            int((task1_df["non_null_years"] == 1).sum()),
            int(task1_long_df[task1_long_df["year"] == 2025].shape[0]),
        ],
    }
)

display(cleaning_summary_df)
display(task1_df.head())
display(task1_analysis_df.head())


# Task 1.2: Derived statistics

## 1.2.3 (by Qingyi Zhu)

### Population-related derived statistics
This section creates five derived population statistics from cleaned Task 1 data.


In [ ]:
population_codes = {
    "ERP_P_20": "total_population",
    "ERP_21": "population_density",
    "ERP_M_20": "male_population",
    "ERP_F_20": "female_population",
    "ERP_18": "working_age_population",
}

population_base_wide = (
    task1_analysis_df.loc[
        task1_analysis_df["measure_code"].isin(population_codes),
        ["year", "measure_code", "value"],
    ]
    .pivot(index="year", columns="measure_code", values="value")
    .rename(columns=population_codes)
    .reset_index()
    .sort_values("year")
)
population_base_wide.columns.name = None

display(population_base_wide)


### Statistic 1: Total population growth rate
Compares current-year total population with previous-year total population.

Raw data: `ERP_P_20`

Formula: `(current population - previous population) / previous population × 100`

Interpretation: Positive means population increased; negative means population decreased.


In [ ]:
population_growth_df = population_base_wide[["year", "total_population"]].copy()
population_growth_df["annual_population_change"] = population_growth_df["total_population"].diff()
population_growth_df["population_growth_rate"] = population_growth_df["total_population"].pct_change() * 100

display(population_growth_df)


### Statistic 2: Population density and implied area
Combines total population with population density to estimate implied land area.

Raw data: `ERP_P_20`, `ERP_21`

Formula: `implied_area = total population / population density`

Interpretation: A stable implied area suggests the population and density indicators are consistent.


In [ ]:
population_density_area_df = population_base_wide[[
    "year", "total_population", "population_density"
]].copy()
population_density_area_df["implied_area_sqkm"] = (
    population_density_area_df["total_population"] / population_density_area_df["population_density"]
)

display(population_density_area_df)


### Statistic 3: Male-to-female population ratio
Compares male population with female population.

Raw data: `ERP_M_20`, `ERP_F_20`

Formula: `male population / female population × 100`

Interpretation: A value close to 100 means the male and female populations are balanced.


In [ ]:
gender_population_df = population_base_wide[[
    "year", "male_population", "female_population"
]].copy()
gender_population_df["males_per_100_females"] = (
    gender_population_df["male_population"] / gender_population_df["female_population"] * 100
)

display(gender_population_df)


### Statistic 4: Working-age dependency ratio
Compares non-working-age population with working-age population.

Raw data: `ERP_P_20`, `ERP_18`

Formula: `(total population - working-age population) / working-age population × 100`

Interpretation: A higher value means more non-working-age people per 100 working-age people.


In [ ]:
dependency_ratio_df = population_base_wide[[
    "year", "total_population", "working_age_population"
]].copy()
dependency_ratio_df["non_working_age_population"] = (
    dependency_ratio_df["total_population"] - dependency_ratio_df["working_age_population"]
)
dependency_ratio_df["dependency_ratio"] = (
    dependency_ratio_df["non_working_age_population"]
    / dependency_ratio_df["working_age_population"]
    * 100
)

display(dependency_ratio_df)


### Statistic 5: Ageing index
Compares seniors aged 65+ with children aged 0-14.

Raw data: `ERP_P_15`-`ERP_P_19`, `ERP_P_2`-`ERP_P_4`

Formula: `seniors 65+ / children 0-14 × 100`

Interpretation: A higher value means the population structure is older.


In [ ]:
children_codes = ["ERP_P_2", "ERP_P_3", "ERP_P_4"]
senior_codes = ["ERP_P_15", "ERP_P_16", "ERP_P_17", "ERP_P_18", "ERP_P_19"]

children = (
    task1_analysis_df[task1_analysis_df["measure_code"].isin(children_codes)]
    .groupby("year")["value"]
    .sum()
)
seniors = (
    task1_analysis_df[task1_analysis_df["measure_code"].isin(senior_codes)]
    .groupby("year")["value"]
    .sum()
)

ageing_index_df = pd.DataFrame({
    "children_0_14": children,
    "seniors_65_plus": seniors,
}).reset_index()
ageing_index_df["ageing_index"] = (
    ageing_index_df["seniors_65_plus"] / ageing_index_df["children_0_14"] * 100
)

display(ageing_index_df)


# Task 2: Parramatta POI collection

The selected area is Parramatta, represented by `SA4_CODE = 125` (`Sydney - Parramatta`). The code below defines a `nearbyPOI(...)` wrapper around the NSW POI API function and contains an explicit loop across every SA2 in the selected SA4. By default, the notebook reads the already saved group database so it can run reproducibly without making new API calls. Set `REFRESH_FROM_API = True` to rebuild the Parramatta-only database from the API loop.

In [ ]:
import importlib
import sqlite3

from shapely.geometry import Point

import task2_sa2

importlib.reload(task2_sa2)

selected_area_name = "QingyiZhu_Parramatta"
selected_sa4_code = "125"
selected_sa4_name = "Sydney - Parramatta"

parramatta_db_path = current_dir / "data" / "qingyizhu_parramatta_task2_poi.db"
source_group_db_path = current_dir / "data" / "task2_poi.db"

REFRESH_FROM_API = False

available_sa4s = task2_sa2.list_nsw_sa4s()
selected_sa4_df = available_sa4s[available_sa4s["SA4_CODE"] == selected_sa4_code]

display(selected_sa4_df)

In [ ]:
def nearbyPOI(xmin, ymin, xmax, ymax):
    return task2_sa2.get_poi_by_bbox(xmin, ymin, xmax, ymax)


def _parramatta_columns(sa2_bbox_df, poi_df):
    sa2_columns = [
        "area_name",
        "sa4_code",
        "sa4_name",
        "sa2_main",
        "sa2_name",
        "state_name",
        "area_sqkm",
        "bbox_xmin",
        "bbox_ymin",
        "bbox_xmax",
        "bbox_ymax",
    ]
    poi_columns = [
        "objectid",
        "poiname",
        "poitype",
        "poigroup",
        "longitude",
        "latitude",
        "area_name",
        "sa4_code",
        "sa4_name",
        "sa2_main",
        "sa2_name",
    ]
    return sa2_bbox_df[sa2_columns].copy(), poi_df[poi_columns].copy()


def collect_parramatta_from_api():
    sa2_bbox_df = task2_sa2.get_sa2_bbox_df_by_code(selected_sa4_code).copy()
    sa2_bbox_df["area_name"] = selected_area_name

    sa2_features = {
        feature["SA2_CODE21"]: feature
        for feature in task2_sa2._load_sa2_features()
        if feature["SA4_CODE21"] == selected_sa4_code
        and feature["STE_NAME21"] == "New South Wales"
        and not feature["geometry"].is_empty
    }

    poi_tables = []
    count_rows = []

    for row in sa2_bbox_df.itertuples(index=False):
        raw_poi_df = nearbyPOI(row.bbox_xmin, row.bbox_ymin, row.bbox_xmax, row.bbox_ymax)
        filtered_poi_df = raw_poi_df.iloc[0:0].copy()

        if not raw_poi_df.empty:
            sa2_geometry = sa2_features[row.sa2_main]["geometry"]
            inside_sa2 = [
                sa2_geometry.covers(Point(poi.longitude, poi.latitude))
                for poi in raw_poi_df.itertuples(index=False)
            ]
            filtered_poi_df = raw_poi_df.loc[inside_sa2].copy()

        if not filtered_poi_df.empty:
            filtered_poi_df["area_name"] = selected_area_name
            filtered_poi_df["sa4_code"] = row.sa4_code
            filtered_poi_df["sa4_name"] = row.sa4_name
            filtered_poi_df["sa2_main"] = row.sa2_main
            filtered_poi_df["sa2_name"] = row.sa2_name
            poi_tables.append(filtered_poi_df)

        count_rows.append(
            {
                "sa2_main": row.sa2_main,
                "sa2_name": row.sa2_name,
                "poi_count": len(filtered_poi_df),
            }
        )

    poi_df = pd.concat(poi_tables, ignore_index=True) if poi_tables else pd.DataFrame(
        columns=[
            "objectid",
            "poiname",
            "poitype",
            "poigroup",
            "longitude",
            "latitude",
            "area_name",
            "sa4_code",
            "sa4_name",
            "sa2_main",
            "sa2_name",
        ]
    )
    poi_count_df = pd.DataFrame(count_rows).sort_values("poi_count", ascending=False)
    sa2_to_save, poi_to_save = _parramatta_columns(sa2_bbox_df, poi_df)
    task2_sa2.save_to_sqlite(sa2_to_save, poi_to_save, str(parramatta_db_path))
    return sa2_to_save, poi_to_save, poi_count_df


def load_parramatta_from_existing_database():
    with sqlite3.connect(source_group_db_path) as conn:
        sa2_bbox_df = pd.read_sql_query(
            "select * from sa2_bbox where sa4_code = ? order by sa2_main",
            conn,
            params=[selected_sa4_code],
        )
        poi_df = pd.read_sql_query(
            """
            select
                objectid,
                poiname,
                poitype,
                poigroup,
                longitude,
                latitude,
                area_name,
                sa4_code,
                sa4_name,
                sa2_main,
                sa2_name
            from poi
            where sa4_code = ?
            order by sa2_main, poi_id
            """,
            conn,
            params=[selected_sa4_code],
        )

    sa2_bbox_df["area_name"] = selected_area_name
    poi_df["area_name"] = selected_area_name
    poi_count_df = (
        poi_df.groupby(["sa2_main", "sa2_name"])
        .size()
        .reset_index(name="poi_count")
        .sort_values("poi_count", ascending=False)
    )
    sa2_to_save, poi_to_save = _parramatta_columns(sa2_bbox_df, poi_df)
    task2_sa2.save_to_sqlite(sa2_to_save, poi_to_save, str(parramatta_db_path))
    return sa2_to_save, poi_to_save, poi_count_df


if REFRESH_FROM_API:
    parramatta_sa2_bbox_df, parramatta_poi_df, parramatta_poi_count_df = collect_parramatta_from_api()
else:
    parramatta_sa2_bbox_df, parramatta_poi_df, parramatta_poi_count_df = load_parramatta_from_existing_database()

display(parramatta_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(parramatta_poi_df.head())
display(parramatta_poi_count_df)

In [ ]:
schema_df = task2_sa2.read_sql(
    "select name, sql from sqlite_master where type = 'table' order by name",
    str(parramatta_db_path),
)
join_df = task2_sa2.read_sql(
    """
    select
        p.poi_id,
        p.area_name,
        p.poiname,
        p.poitype,
        s.sa4_code,
        s.sa4_name,
        s.sa2_name
    from poi p
    join sa2_bbox s
        on p.sa2_main = s.sa2_main
    limit 20
    """,
    str(parramatta_db_path),
)

display(schema_df)
display(join_df)

# Task 3: Parramatta SQL score calculation

This section calculates a well-resourced score for Parramatta SA2s using SQL. The basic score uses POI counts, standardises them with a z-score, and applies the sigmoid function:

$$Score = S(z_{POI}) = \frac{1}{1 + e^{-z_{POI}}}$$

The rubric's population rule is applied by joining the Parramatta SA2 population file and excluding SA2s where `population_2022 < 100`. The extension adds an area-adjusted POI density score, because raw POI counts can favour geographically larger SA2s.

In [ ]:
import numpy as np

population_path = current_dir / "data" / "qingyizhu_parramatta_population_2022.csv"
parramatta_population_df = pd.read_csv(population_path, dtype={"sa2_main": str})

with sqlite3.connect(parramatta_db_path) as conn:
    conn.execute("drop table if exists sa2_population")
    conn.execute(
        """
        create table sa2_population (
            sa2_main text primary key,
            sa2_name text,
            population_2022 integer
        )
        """
    )
    parramatta_population_df.to_sql("sa2_population", conn, if_exists="append", index=False)
    conn.execute("create index if not exists idx_sa2_population_pop on sa2_population(population_2022)")

    conn.executescript(
        """
        drop table if exists qingyi_sa2_scores;

        create table qingyi_sa2_scores as
        with counts as (
            select
                s.area_name,
                s.sa4_code,
                s.sa4_name,
                s.sa2_main,
                s.sa2_name,
                s.area_sqkm,
                pop.population_2022,
                count(p.poi_id) as poi_count,
                count(p.poi_id) * 1.0 / nullif(s.area_sqkm, 0) as poi_density_sqkm
            from sa2_bbox s
            join sa2_population pop
                on s.sa2_main = pop.sa2_main
            left join poi p
                on s.sa2_main = p.sa2_main
            where pop.population_2022 >= 100
            group by
                s.area_name,
                s.sa4_code,
                s.sa4_name,
                s.sa2_main,
                s.sa2_name,
                s.area_sqkm,
                pop.population_2022
        ),
        stats as (
            select
                avg(poi_count * 1.0) as mean_poi,
                case
                    when count(*) > 1 then sqrt(
                        (sum(poi_count * poi_count * 1.0)
                        - sum(poi_count * 1.0) * sum(poi_count * 1.0) / count(*))
                        / (count(*) - 1)
                    )
                    else 0
                end as stddev_poi,
                avg(poi_density_sqkm * 1.0) as mean_density,
                case
                    when count(*) > 1 then sqrt(
                        (sum(poi_density_sqkm * poi_density_sqkm * 1.0)
                        - sum(poi_density_sqkm * 1.0) * sum(poi_density_sqkm * 1.0) / count(*))
                        / (count(*) - 1)
                    )
                    else 0
                end as stddev_density
            from counts
        ),
        zscores as (
            select
                c.*,
                case
                    when s.stddev_poi is null or s.stddev_poi = 0 then 0
                    else (c.poi_count - s.mean_poi) / s.stddev_poi
                end as z_poi,
                case
                    when s.stddev_density is null or s.stddev_density = 0 then 0
                    else (c.poi_density_sqkm - s.mean_density) / s.stddev_density
                end as z_density
            from counts c
            cross join stats s
        )
        select
            area_name,
            sa4_code,
            sa4_name,
            sa2_main,
            sa2_name,
            area_sqkm,
            population_2022,
            poi_count,
            poi_density_sqkm,
            z_poi,
            1.0 / (1.0 + exp(-z_poi)) as score,
            z_density,
            1.0 / (1.0 + exp(-(0.7 * z_poi + 0.3 * z_density))) as extended_score
        from zscores;

        create unique index if not exists idx_qingyi_scores_sa2
            on qingyi_sa2_scores(sa2_main);
        create index if not exists idx_qingyi_scores_score
            on qingyi_sa2_scores(score);
        """
    )

    score_df = pd.read_sql_query(
        "select * from qingyi_sa2_scores order by score desc",
        conn,
    )
    filtered_out_df = pd.read_sql_query(
        """
        select
            s.sa2_main,
            s.sa2_name,
            pop.population_2022
        from sa2_bbox s
        join sa2_population pop
            on s.sa2_main = pop.sa2_main
        where pop.population_2022 < 100
        order by pop.population_2022, s.sa2_main
        """,
        conn,
    )
    score_summary_df = pd.read_sql_query(
        """
        select
            sa4_code,
            sa4_name,
            count(*) as retained_sa2_count,
            sum(poi_count) as total_poi,
            round(avg(score), 6) as mean_score,
            round(min(score), 6) as min_score,
            round(max(score), 6) as max_score,
            round(avg(extended_score), 6) as mean_extended_score
        from qingyi_sa2_scores
        group by sa4_code, sa4_name
        """,
        conn,
    )

score_output_path = current_dir / "data" / "qingyizhu_parramatta_task3_scores.csv"
score_df.to_csv(score_output_path, index=False)

display(score_summary_df)
display(filtered_out_df)
display(score_df.head(10))

## Task 3 interpretation

The population filter removes very small-resident-population SA2s before scoring, which prevents industrial or special-purpose zones from distorting the Parramatta comparison. The base score ranks SA2s by POI concentration relative to the selected SA4. The extension keeps the POI-count signal as the main component but adds POI density per square kilometre, so compact SA2s with many services are not understated compared with larger SA2s.